# Module 8 - Session 5: Practical Exercises

**Total Time Estimate:** 90-120 minutes

**Objective:** To gain hands-on experience using a pre-trained YOLO model to perform object detection on images.

## Setup

Please use Python with OpenCV and NumPy.

```bash
pip install opencv-python numpy
```

For Exercise 4, you will also need the Ultralytics package:

```bash
pip install ultralytics
```

## YOLOv3 Files

You will need to download three files for Exercises 2 and 3:

1. `yolov3.cfg` - the model configuration file,
2. `yolov3.weights` - the pre-trained model weights,
3. `coco.names` - the class names file.

You can find these online by searching for `yolov3.weights download`. The official source is on the YOLO author's website. For convenience, many tutorials also provide direct links.

Place these files in the same folder as this notebook, or update the file paths in your code.

---

## Exercise 1: Conceptual Questions (30 minutes)

### Foundation

YOLO is a single-shot object detector. Instead of first proposing regions and then classifying them, YOLO predicts bounding boxes and class probabilities directly from the image in one forward pass.

This makes YOLO fast, but it also introduces important design choices such as anchor boxes, confidence thresholds, and Non-Max Suppression.

### Build

Answer the following questions in markdown cells or code comments.

1. **Anchor Boxes**

   YOLO uses anchor boxes, which are pre-defined bounding box shapes, such as a tall thin box for a person or a wide short box for a car.

   Why is this a better approach than having the network predict the box shape from scratch every time? How do anchor boxes help the network specialize?

2. **NMS Threshold**

   The IoU threshold in Non-Max Suppression is a critical hyperparameter.

   What would happen if you set this threshold very high, such as `0.9`? What would happen if you set it very low, such as `0.1`?

3. **The Single-Shot Trade-off**

   Single-shot detectors like YOLO are generally less accurate than two-stage detectors like Faster R-CNN, especially for very small objects.

   Based on your understanding of the architectures, why do you think this is the case? Think about the two-stage process of proposing regions and then refining them.

### Result

Your answers should show that you understand how YOLO balances speed and accuracy, and why post-processing choices affect the final detections.

---

## Exercise 2: Object Detection on an Image with YOLO (60 minutes)

### Foundation

OpenCV's DNN module can load a pre-trained YOLOv3 network and run object detection without using a deep learning framework directly. The model produces many candidate detections, which must be filtered by confidence and then cleaned up with Non-Max Suppression.

### Build

Write a script or notebook cells to run a pre-trained YOLOv3 model on a single image.

1. **Load Model and Class Names**

   - Load the class names from `coco.names` into a list.
   - Load the YOLO network using OpenCV's DNN module:

     ```python
     net = cv2.dnn.readNet('yolov3.weights', 'yolov3.cfg')
     ```

2. **Load and Prepare Image**

   - Load a sample image of your choice that contains objects from the COCO dataset, such as people, cars, dogs, or bicycles.
   - Get the image's height and width.
   - Create a blob from your image:

     ```python
     blob = cv2.dnn.blobFromImage(
         image,
         1 / 255.0,
         (416, 416),
         swapRB=True,
         crop=False,
     )
     ```

3. **Perform a Forward Pass**

   - Set the input to the network:

     ```python
     net.setInput(blob)
     ```

   - Get the names of the output layers:

     ```python
     output_layer_names = net.getUnconnectedOutLayersNames()
     ```

   - Run the forward pass to get the detections:

     ```python
     layer_outputs = net.forward(output_layer_names)
     ```

4. **Process the Detections**

   - Loop through `layer_outputs`.
   - Inside that loop, loop through each detection.
   - For each detection, extract the class ID and confidence score.
   - If the confidence is above a threshold such as `0.5`, save the bounding box coordinates, confidence, and class ID.

5. **Apply Non-Max Suppression**

   After you have a list of good detections, use OpenCV's built-in NMS function to remove duplicates:

   ```python
   indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
   ```

6. **Draw the Bounding Boxes**

   - Loop through the final indexes returned by NMS.
   - For each one, get the corresponding box, label, and confidence.
   - Use `cv2.rectangle()` and `cv2.putText()` to draw the final bounding boxes and labels on your original image.
   - Display the final image.

### Result

Your notebook should show the original image and the final detected image with bounding boxes, class labels, and confidence scores.

---

## Exercise 3: Challenge Problem - Real-Time Webcam Detection

### Foundation

Running detection on a video stream applies the same model pipeline repeatedly to each frame. The challenge is performance: each frame must be processed quickly enough for the output to feel real-time.

### Build

Modify your Exercise 2 script to read from a webcam instead of a static image.

1. Use `cv2.VideoCapture(0)` to open your webcam.

   ```python
   cap = cv2.VideoCapture(0)
   ```

2. Place the full detection process inside a `while True:` loop that reads frames from the webcam.

3. For each frame:

   - create the blob,
   - run the forward pass,
   - process detections,
   - apply NMS,
   - draw the final boxes.

4. Display the resulting frame in a window using `cv2.imshow()`.

5. Add a way to break the loop, such as pressing the `q` key.

6. Release the camera and close OpenCV windows when finished.

### Analysis

In a markdown cell, answer the following questions:

- How fast does detection run on your computer?
- Is it truly real-time?
- What happens to the performance if you increase the blob input size from `(416, 416)` to `(608, 608)`?
- What trade-off do you observe between detection quality and speed?

### Result

Your submission should include the webcam detection code and a short written performance analysis.

---

## Exercise 4: Try the Latest Version - YOLO26

### Foundation

The YOLO family continues to evolve. According to the current Ultralytics YOLO26 documentation, YOLO26 supports object detection using model files such as `yolo26n.pt`, `yolo26s.pt`, `yolo26m.pt`, `yolo26l.pt`, and `yolo26x.pt`.

The documentation also shows that a COCO-pretrained YOLO26 detection model can be loaded with the Ultralytics Python API:

```python
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
results = model('path/to/image.jpg')
```

Reference: https://docs.ultralytics.com/models/yolo26/#usage-example

### Build

Run a prediction using the object detection version of YOLO26 and compare it with your YOLOv3 result from Exercise 2.

1. **Install Ultralytics**

   ```bash
   pip install ultralytics
   ```

2. **Load a YOLO26 Detection Model**

   Start with the nano model because it is small and fast:

   ```python
   from ultralytics import YOLO

   model = YOLO('yolo26n.pt')
   ```

3. **Run Prediction on the Same Image**

   Use the same image from Exercise 2 so the comparison is fair.

   ```python
   results = model.predict('your_image.jpg', imgsz=640, conf=0.5)
   results[0].show()
   ```

4. **Save or Display the Result**

   You may save the prediction output using:

   ```python
   results = model.predict('your_image.jpg', imgsz=640, conf=0.5, save=True)
   ```

5. **Measure Runtime**

   Time both YOLOv3 and YOLO26 on the same image. You can use `time.perf_counter()` around each prediction block.

6. **Compare Results**

   In a markdown cell, answer:

   - Is the detection result better than with YOLOv3?
   - Is the time performance higher or lower?
   - Which objects were detected by both models?
   - Which objects were missed by one model but detected by the other?
   - What is your hypothesis for why there is a difference?

### Result

Your final comparison should include the YOLOv3 output image, the YOLO26 output image, the runtime for both models, and a short explanation of the difference in detection quality and speed.

---

## Submission Checklist

Before submitting, make sure your notebook includes:

- written answers for all conceptual questions in Exercise 1,
- working YOLOv3 image detection code,
- a displayed or saved image with YOLOv3 bounding boxes and labels,
- working webcam detection code or a clear note if your environment does not support webcam access,
- a short analysis of webcam speed and input-size trade-offs,
- working YOLO26 prediction code using the Ultralytics API,
- a side-by-side comparison of YOLOv3 and YOLO26 results,
- runtime measurements for both models,
- a short hypothesis explaining any difference in quality or speed.